# 01 · Single-resolution NF — stage by stage

The quickstart notebook runs everything end-to-end as one call. This notebook
instead **invokes each stage individually** so you can see what each step
does (and inspect the intermediate files).

Single-resolution = `NumLoops = 0` (no `GridRefactor` key) → loop 0 only:
denoise (optional) → preprocessing → image processing → fitting → parse mic
→ consolidate.

Use this notebook when:

- you're debugging or tuning one stage
- you want to swap a stage with a custom implementation
- you want to skip a stage that already ran

In [ ]:
import os, shutil, tempfile, time
from pathlib import Path
from argparse import Namespace

import h5py
import numpy as np

from midas_nf_pipeline import stages
from midas_nf_pipeline.params import parse_parameters, update_param_file
from midas_nf_pipeline.parse_mic import ParseMicParams, parse_mic
from midas_nf_pipeline.consolidate import generate_consolidated_hdf5

In [ ]:
MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or os.environ.get('MIDAS_INSTALL_DIR') or '.')
AU_DIR = MIDAS_HOME / 'NF_HEDM/Example/sim'

ws = Path(tempfile.mkdtemp(prefix='nf_single_res_'))
param = ws / 'test_ps_au.txt'
shutil.copy2(AU_DIR / 'test_ps_au.txt', param)
# strip GridRefactor → single-resolution
text = param.read_text()
lines = [l for l in text.splitlines() if not l.strip().startswith('GridRefactor')]
lines.append(f'OutputDirectory {ws}')
param.write_text('\n'.join(lines) + '\n')
print('workspace:', ws)

p = parse_parameters(param)
p['resultFolder'] = str(ws)
p['logDir'] = str(ws / 'midas_log')
p['nCPUs'] = 4
(ws / 'midas_log').mkdir(exist_ok=True)
os.chdir(ws)

## Stage 1a — Generate `hkls.csv` (midas-hkls)

In [ ]:
stages.run_get_hkls(p, param)
print((ws / 'hkls.csv').read_text().splitlines()[:6])

## Stage 1b — Seed orientations (cache)

In [ ]:
if not p.get('SeedOrientations') or not Path(p['SeedOrientations']).exists():
    stages.run_seed_orientations_from_cache(p, install_dir=str(MIDAS_HOME))
print('SeedOrientations:', p['SeedOrientations'])

## Stage 1c — Hex grid + optional tomo / mask filter

In [ ]:
stages.run_hex_grid(p, param)
stages.run_tomo_filter(p)
stages.run_grid_mask(p)
print((ws / 'grid.txt').read_text().splitlines()[:3])

## Stage 1d — Diffraction-spot simulation

In [ ]:
# Update NrOrientations on disk so the C-format readers in fit_orientation see it
with open(p['SeedOrientations']) as f:
    n_orient = sum(1 for _ in f)
update_param_file(param, {'NrOrientations': str(n_orient)})
stages.run_diffr_spots(p, param)
for fn in ('DiffractionSpots.bin', 'OrientMat.bin', 'Key.bin'):
    print(fn, '→', (ws / fn).stat().st_size, 'bytes')

## Stage 2 — Image processing (median + peak detection per detector distance)

In [ ]:
t0 = time.time()
stages.run_image_processing(p, param)
print(f'image processing took {time.time()-t0:.1f}s')
print('SpotsInfo.bin →', (ws / 'SpotsInfo.bin').stat().st_size, 'bytes')

## Stage 3 — Orientation fitting (midas-nf-fitorientation)

In [ ]:
t0 = time.time()
stages.run_fitting(p, param, n_cpus=p['nCPUs'], device='auto')
print(f'fitting took {time.time()-t0:.1f}s')

## Stage 4 — ParseMic (binary → text + maps)

In [ ]:
out = stages.run_parse_mic(p)
print('outputs:', list(out.keys()))

## Stage 5 — Consolidate everything into a single H5

In [ ]:
with open(param) as f: param_text = f.read()
h5 = stages.run_consolidate(p, param_text=param_text)
print('H5 →', h5)
with h5py.File(h5, 'r') as fp:
    print('Top-level keys:', list(fp.keys()))